# Flowers CNN Exercise

This notebook is based on `ex_flowers_cnn.ipynb`, but several key pieces have been replaced with `TODO` markers. Fill them in to rebuild the full training pipeline yourself.

Suggested order:
1. Complete the dataset utilities.
2. Implement the CNN architecture.
3. Finish the training and evaluation loops.
4. Run `main()` once the TODOs are complete.


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# NOTE: you will need these packages on top of the defaults
# pip install torch torchvision datasets pillow matplotlib 


## Configuration

You can usually leave these settings as they are at first. The `n_pix` value being at e.g. 32 is reasonable for prototyping.


In [ ]:
@dataclass
class Config:
    dataset_name: str = "nkirschi/oxford-flowers"
    image_size: int = 32
    batch_size: int = 64
    epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    num_workers: int = 0
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir: str | None = "flowers_cnn_out/"


## Dataset Utilities

In [ ]:
def make_splits_if_needed(ds_dict, seed):
    """
    Some HF datasets already have train/validation/test splits.
    If not, create them from a single split.
    """
    split_names = set(ds_dict.keys())

    if {"train", "validation", "test"}.issubset(split_names):
        return ds_dict

    if "train" in split_names and "test" in split_names and "validation" not in split_names:
        temp = ds_dict["train"].train_test_split(test_size=0.1, seed=seed)
        return {
            "train": temp["train"],
            "validation": temp["test"],
            "test": ds_dict["test"],
        }

    # Fallback: assume there is one split only
    first_key = list(ds_dict.keys())[0]
    full = ds_dict[first_key]
    train_val_test = full.train_test_split(test_size=0.2, seed=seed)
    val_test = train_val_test["test"].train_test_split(test_size=0.5, seed=seed)

    return {
        "train": train_val_test["train"],
        "validation": val_test["train"],
        "test": val_test["test"],
    }


def infer_label_column(example):
    """
    Tries to infer the label column name.
    Common possibilities: 'label', 'labels'
    """
    if "label" in example:
        return "label"
    if "labels" in example:
        return "labels"
    raise KeyError("Could not find label column. Expected 'label' or 'labels'.")


class HFDatasetWrapper(Dataset):
    def __init__(self, hf_split, image_transform=None, label_column="label"):
        self.hf_split = hf_split
        self.image_transform = image_transform
        self.label_column = label_column

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = item["image"]
        label = item[self.label_column]

        # Defensive conversion in case image is not already PIL
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))

        image = image.convert("RGB")

        if self.image_transform is not None:
            image = self.image_transform(image)

        return image, int(label)


## Visualisation

These helper functions are provided so you can focus on the core modelling code.


In [ ]:
def get_out_dir(out_dir_name):
    if out_dir_name is None:
        return None

    out_dir = Path(out_dir_name)
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def show_examples(hf_split, label_column, n=9, out_dir=None):
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    axes = axes.ravel()

    indices = np.random.choice(len(hf_split), size=n, replace=False)

    for ax, idx in zip(axes, indices):
        item = hf_split[int(idx)]
        ax.imshow(item["image"])
        ax.set_title(f"class = {item[label_column]}")
        ax.axis("off")

    fig.tight_layout()
    if out_dir is not None:
        fig.savefig(out_dir / "examples.png")
    else:
        plt.show()


def show_predictions(model, hf_split, label_column, image_transform, n=9, device="cpu", out_dir=None):
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    axes = axes.ravel()

    n = min(n, len(axes), len(hf_split))
    indices = np.random.choice(len(hf_split), size=n, replace=False)

    class_names = None
    features = getattr(hf_split, "features", None)
    if features is not None and label_column in features:
        class_names = getattr(features[label_column], "names", None)

    images = []
    targets = []
    for idx in indices:
        item = hf_split[int(idx)]
        image = item["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        images.append(image.convert("RGB"))
        targets.append(int(item[label_column]))

    was_training = model.training
    model.eval()
    with torch.no_grad():
        batch = torch.stack([image_transform(image) for image in images]).to(device)
        preds = model(batch).argmax(dim=1).cpu().tolist()
    model.train(was_training)

    def format_label(label_idx):
        if class_names is None:
            return str(label_idx)
        return str(class_names[label_idx])

    for ax, image, pred, target in zip(axes, images, preds, targets):
        ax.imshow(image)
        ax.set_title(
            f"pred = {format_label(pred)}\ntrue = {format_label(target)}",
            color="green" if pred == target else "red",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.tight_layout()
    if out_dir is not None:
        fig.savefig(out_dir / "predictions.png")
    plt.show()


def init_training_curve_plots(out_dir=None):
    if out_dir is None:
        plt.ion()

    plot_specs = {
        "loss": ("Loss", "train_loss", "val_loss"),
        "accuracy": ("Accuracy", "train_acc", "val_acc"),
    }
    plots = {}

    for name, (ylabel, train_key, val_key) in plot_specs.items():
        fig, ax = plt.subplots(figsize=(7, 4))
        (train_line,) = ax.plot([], [], label=f"train {name}")
        (val_line,) = ax.plot([], [], label=f"val {name}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        fig.tight_layout()

        if out_dir is None:
            fig.show()

        plots[name] = {
            "fig": fig,
            "ax": ax,
            "train_line": train_line,
            "val_line": val_line,
            "train_key": train_key,
            "val_key": val_key,
        }

    return plots


def update_training_curve_plots(plots, history, out_dir=None):
    epochs = np.arange(1, len(history["train_loss"]) + 1)

    for name, plot in plots.items():
        train_values = history[plot["train_key"]]
        val_values = history[plot["val_key"]]
        all_values = train_values + val_values

        plot["train_line"].set_data(epochs, train_values)
        plot["val_line"].set_data(epochs, val_values)

        plot["ax"].set_xlim(1, max(2, len(epochs)))
        y_min = min(all_values)
        y_max = max(all_values)
        y_pad = max(0.01, (y_max - y_min) * 0.1)
        plot["ax"].set_ylim(y_min - y_pad, y_max + y_pad)

        plot["fig"].tight_layout()
        if out_dir is not None:
            plot["fig"].savefig(out_dir / f"{name}.png")
        else:
            plot["fig"].canvas.draw_idle()
            plot["fig"].canvas.flush_events()

    if out_dir is None:
        plt.pause(0.001)


## Model

In [ ]:
class SmallFlowerCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


## Training and Evaluation

Finish the minibatch training loop and the evaluation pass.


In [ ]:
def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for x, y in loader:
        # TODO: move the batch to the target device.
        # TODO: run the model, compute the loss, and update parameters when training.
        # TODO: accumulate loss, correct predictions, and number of examples.
        raise NotImplementedError("Implement one training/evaluation epoch.")

    mean_loss = total_loss / total_examples
    mean_acc = total_correct / total_examples
    return mean_loss, mean_acc


def evaluate(model, loader, device="cpu"):
    model.eval()
    preds_all = []
    targets_all = []

    with torch.no_grad():
        for x, y in loader:
            # TODO: move inputs to the device, run the model,
            # and collect predictions and targets.
            raise NotImplementedError("Implement the evaluation loop.")

    preds_all = torch.cat(preds_all)
    targets_all = torch.cat(targets_all)
    acc = (preds_all == targets_all).float().mean().item()
    return acc, preds_all, targets_all


## Main Function

The overall training pipeline is provided here so you can plug your implementations into it.


In [ ]:
def main():
    cfg = Config()
    print(f"Using device: {cfg.device}")
    out_dir = get_out_dir(cfg.out_dir)

    # Load dataset
    ds = load_dataset(cfg.dataset_name)
    ds = make_splits_if_needed(ds, cfg.seed)

    # Infer label column
    example_item = ds["train"][0]
    label_column = infer_label_column(example_item)

    # Infer number of classes
    all_train_labels = ds["train"][label_column]
    num_classes = int(max(all_train_labels)) + 1

    print("Splits:")
    for split_name in ds:
        print(f"  {split_name}: {len(ds[split_name])}")

    print(f"Detected label column: {label_column}")
    print(f"Detected number of classes: {num_classes}")

    # Optional quick visualisation
    show_examples(ds["train"], label_column, out_dir=out_dir)

    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5],
                             std=[0.5, 0.5, 0.5]),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5],
                             std=[0.5, 0.5, 0.5]),
    ])

    train_data = HFDatasetWrapper(ds["train"], image_transform=train_transform, label_column=label_column)
    val_data = HFDatasetWrapper(ds["validation"], image_transform=eval_transform, label_column=label_column)
    test_data = HFDatasetWrapper(ds["test"], image_transform=eval_transform, label_column=label_column)

    train_loader = DataLoader(
        train_data,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_data,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_data,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    model = SmallFlowerCNN(num_classes=num_classes).to(cfg.device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    best_val_acc = -1.0
    best_state = None

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }
    training_plots = init_training_curve_plots(out_dir=out_dir)

    for epoch in range(cfg.epochs):
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer=optimizer, device=cfg.device
        )

        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer=None, device=cfg.device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch {epoch+1:02d}/{cfg.epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )
        update_training_curve_plots(training_plots, history, out_dir=out_dir)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_acc, preds, targets = evaluate(model, test_loader, device=cfg.device)
    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    show_predictions(
        model,
        ds["test"],
        label_column,
        eval_transform,
        device=cfg.device,
        out_dir=out_dir,
    )
    if out_dir is not None:
        for plot in training_plots.values():
            plt.close(plot["fig"])


In [ ]:
main()
